In [40]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

project_root = Path.cwd().parents[2]
sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai


In [41]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parents[2]

print("Project root:", project_root)
# Load the saved student assignments
split_path = project_root  / "data" / "processed" / "split_assignments.csv"

split_assignments = pd.read_csv(split_path)

# Keep only training-pool students
train_assignments = split_assignments[
    split_assignments["partition"] == "train_pool"
].copy()

# Convert fold numbers to integers
train_assignments["validation_fold"] = (
    train_assignments["validation_fold"].astype(int)
)

# Extract training student IDs
train_student_ids = set(train_assignments["user_id"])

print("Training students:", len(train_student_ids))
print("\nStudents assigned to each validation fold:")
print(train_assignments["validation_fold"].value_counts().sort_index())

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Training students: 3373

Students assigned to each validation fold:
validation_fold
1    675
2    675
3    675
4    674
5    674
Name: count, dtype: int64


In [42]:
data_path = (
    project_root
    / "data"
    / "processed"
    / "main_questions_cleaned.csv"
)

train_chunks = []

for chunk in pd.read_csv(data_path, chunksize=50000):

    training_rows = chunk[
        chunk["user_id"].isin(train_student_ids)
    ]

    train_chunks.append(training_rows)

# Combine all retained training interactions
train_df = pd.concat(train_chunks, ignore_index=True)

# Sort interactions chronologically within each student
train_df = (
    train_df.sort_values(["user_id", "order_id"])
            .reset_index(drop=True)
)

print("Training data shape:", train_df.shape)
print("Unique training students:", train_df["user_id"].nunique())
print("Missing values:\n", train_df.isnull().sum())

Training data shape: (219983, 8)
Unique training students: 3373
Missing values:
 user_id              0
order_id             0
problem_id           0
correct              0
skill_id         13449
skill_set            0
attempt_count        0
answer_type          0
dtype: int64


In [43]:
# Check whether order_id is increasing within each student
is_chronological = (
    train_df.groupby("user_id")["order_id"]
    .apply(lambda s: s.is_monotonic_increasing)
)

print("All students chronologically sorted:", is_chronological.all())

# Count students whose order_id is not monotonic
print(
    "Students with incorrect ordering:",
    (~is_chronological).sum()
)

# Check duplicate order_id values within each student
duplicate_count = train_df.duplicated(
    subset=["user_id", "order_id"]
).sum()

print("Duplicate student-order pairs:", duplicate_count)

All students chronologically sorted: True
Students with incorrect ordering: 0
Duplicate student-order pairs: 0


In [44]:
fold_datasets = []

for fold_number in range(1, 6):

    # Students belonging to this validation fold
    val_student_ids = train_assignments.loc[
        train_assignments["validation_fold"] == fold_number,
        "user_id"
    ]

    # All remaining training-pool students
    fold_train_student_ids = train_assignments.loc[
        train_assignments["validation_fold"] != fold_number,
        "user_id"
    ]

    # Construct training and validation DataFrames
    fold_train_df = train_df[
        train_df["user_id"].isin(fold_train_student_ids)
    ].copy()

    fold_val_df = train_df[
        train_df["user_id"].isin(val_student_ids)
    ].copy()

    fold_datasets.append((fold_train_df, fold_val_df))

    print(
        f"Fold {fold_number}: "
        f"Training rows = {len(fold_train_df)}, "
        f"Validation rows = {len(fold_val_df)}"
    )

Fold 1: Training rows = 177324, Validation rows = 42659
Fold 2: Training rows = 176273, Validation rows = 43710
Fold 3: Training rows = 170186, Validation rows = 49797
Fold 4: Training rows = 178311, Validation rows = 41672
Fold 5: Training rows = 177838, Validation rows = 42145


In [45]:
for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    train_students = set(fold_train_df["user_id"])
    val_students = set(fold_val_df["user_id"])

    overlap = train_students.intersection(val_students)

    print(
        f"Fold {fold_number}: "
        f"Student overlap = {len(overlap)}"
    )

Fold 1: Student overlap = 0
Fold 2: Student overlap = 0
Fold 3: Student overlap = 0
Fold 4: Student overlap = 0
Fold 5: Student overlap = 0


In [46]:

from src.feature_engineering import add_overall_history_features

v11_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    # Generate training features using only earlier training interactions
    v11_train_df = add_overall_history_features(
        fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate validation features using training data for the global prior
    v11_val_df = add_overall_history_features(
        fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v11_fold_datasets.append(
        (v11_train_df, v11_val_df)
    )

    print(
        f"Fold {fold_number}: "
        f"Training shape = {v11_train_df.shape}, "
        f"Validation shape = {v11_val_df.shape}"
    )


Fold 1: Training shape = (177324, 13), Validation shape = (42659, 13)
Fold 2: Training shape = (176273, 13), Validation shape = (43710, 13)
Fold 3: Training shape = (170186, 13), Validation shape = (49797, 13)
Fold 4: Training shape = (178311, 13), Validation shape = (41672, 13)
Fold 5: Training shape = (177838, 13), Validation shape = (42145, 13)


In [47]:

# Select Fold 1
v11_train_df, v11_val_df = v11_fold_datasets[0]

# Inspect the first five interactions of one training student
sample_student_id = (
    v11_train_df["user_id"].value_counts().loc[
        lambda s: s >= 5
    ].index[0]
)

sample = v11_train_df[
    v11_train_df["user_id"] == sample_student_id
].head(5)

print(
    sample[
        [
            "user_id",
            "order_id",
            "correct",
            "previous_interactions",
            "previous_correct",
            "global_historical_prior",
            "overall_smoothed_accuracy"
        ]
    ].to_string(index=False)
)


 user_id  order_id  correct  previous_interactions  previous_correct  global_historical_prior  overall_smoothed_accuracy
   78970  20765556        1                      0                 0                 0.764496                   0.764496
   78970  20765796        0                      1                 1                 0.764915                   0.786287
   78970  20766021        0                      2                 1                 0.764235                   0.720196
   78970  20766237        0                      3                 1                 0.763556                   0.664274
   78970  20855602        1                      4                 1                 0.748455                   0.606039


In [48]:
# Check training and validation feature columns
print("Training columns:")
print(v11_train_df.columns.tolist())

print("\nValidation columns:")
print(v11_val_df.columns.tolist())

# Check for missing values in the new features
print("\nMissing values:")
print(
    v11_train_df[
        ["global_historical_prior", "overall_smoothed_accuracy"]
    ].isnull().sum()
)

# Check prior range
print("\nGlobal prior range:")
print(
    v11_train_df["global_historical_prior"].min(),
    v11_train_df["global_historical_prior"].max()
)

Training columns:
['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count', 'answer_type', 'previous_interactions', 'previous_correct', 'global_historical_prior', 'overall_smoothed_accuracy', 'log_previous_interactions']

Validation columns:
['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count', 'answer_type', 'previous_interactions', 'previous_correct', 'global_historical_prior', 'overall_smoothed_accuracy', 'log_previous_interactions']

Missing values:
global_historical_prior      0
overall_smoothed_accuracy    0
dtype: int64

Global prior range:
0.5 1.0


In [49]:
feature_columns = ["overall_smoothed_accuracy"]

for fold_number, (train_data, val_data) in enumerate(
    v11_fold_datasets, start=1
):
    X_train = train_data[feature_columns]
    y_train = train_data["correct"]

    X_val = val_data[feature_columns]
    y_val = val_data["correct"]

    print(
        f"Fold {fold_number}: "
        f"X_train = {X_train.shape}, "
        f"X_val = {X_val.shape}"
    )

Fold 1: X_train = (177324, 1), X_val = (42659, 1)
Fold 2: X_train = (176273, 1), X_val = (43710, 1)
Fold 3: X_train = (170186, 1), X_val = (49797, 1)
Fold 4: X_train = (178311, 1), X_val = (41672, 1)
Fold 5: X_train = (177838, 1), X_val = (42145, 1)


In [50]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

v11_models = []
v11_results = []

for fold_number, (train_data, val_data) in enumerate(
    v11_fold_datasets, start=1
):

    X_train = train_data[feature_columns]
    y_train = train_data["correct"]

    X_val = val_data[feature_columns]
    y_val = val_data["correct"]

    # Create preprocessing and model pipeline
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    # Fit scaler and logistic regression using training data only
    model.fit(X_train, y_train)

    # Predict probability of correct outcome
    y_pred = model.predict_proba(X_val)[:, 1]

    # Evaluate validation predictions
    ll = log_loss(y_val, y_pred, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred)

    v11_models.append(model)

    v11_results.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs
    })

    print(
        f"Fold {fold_number}: "
        f"Log Loss = {ll:.4f}, "
        f"Brier Score = {bs:.4f}"
    )

Fold 1: Log Loss = 0.5945, Brier Score = 0.2036
Fold 2: Log Loss = 0.5901, Brier Score = 0.2014
Fold 3: Log Loss = 0.6045, Brier Score = 0.2081
Fold 4: Log Loss = 0.5949, Brier Score = 0.2037
Fold 5: Log Loss = 0.5860, Brier Score = 0.1998


In [51]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Find the project root by locating the folder containing src/
current_path = Path.cwd()

project_root = next(
    (path for path in [current_path, *current_path.parents]
     if (path / "src").is_dir()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate the project root.")

sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai


In [52]:
# Collect out-of-fold predictions for V1.1

v11_predictions = []

for fold_number, (train_data, val_data) in enumerate(
    v11_fold_datasets, start=1
):

    # Retrieve the already-trained model for this fold
    model = v11_models[fold_number - 1]

    # Generate validation probabilities
    X_val = val_data[feature_columns]
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Store actual outcomes and predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_data["user_id"].values,
        "order_id": val_data["order_id"].values,
        "fold": fold_number,
        "y_true": val_data["correct"].values,
        "y_pred_proba": y_pred_proba,
        "model": "V1.1"
    })

    v11_predictions.append(fold_predictions)

# Combine predictions from all five folds
v11_predictions_df = pd.concat(
    v11_predictions,
    ignore_index=True
)

print("Prediction collection completed.")
print("Total predictions:", len(v11_predictions_df))
display(v11_predictions_df.head())

Prediction collection completed.
Total predictions: 219983


,user_id,order_id,fold,y_true,y_pred_proba,model
0,21825,21616385,1,0,0.732273,V1.1
1,21825,21616478,1,1,0.661964,V1.1
2,21825,21616677,1,1,0.692306,V1.1
3,21825,21616919,1,1,0.716755,V1.1
4,21825,21671620,1,0,0.734329,V1.1


In [53]:
print("Total rows:", len(v11_predictions_df))

print("\nMissing values:")
print(v11_predictions_df.isna().sum())

print("\nFold distribution:")
print(v11_predictions_df["fold"].value_counts().sort_index())

print("\nProbability range:")
print(
    v11_predictions_df["y_pred_proba"].min(),
    v11_predictions_df["y_pred_proba"].max()
)

print("\nActual outcome distribution:")
print(v11_predictions_df["y_true"].value_counts())

Total rows: 219983

Missing values:
user_id         0
order_id        0
fold            0
y_true          0
y_pred_proba    0
model           0
dtype: int64

Fold distribution:
fold
1    42659
2    43710
3    49797
4    41672
5    42145
Name: count, dtype: int64

Probability range:
0.07774794781198782 0.9109627105149137

Actual outcome distribution:
y_true
1    144886
0     75097
Name: count, dtype: int64


In [54]:
prediction_dir = (
    project_root / "data" / "processed" / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

v11_prediction_path = prediction_dir / "v1_1_predictions.csv"

v11_predictions_df.to_csv(
    v11_prediction_path,
    index=False
)

print("V1.1 predictions saved successfully.")
print("File:", v11_prediction_path)

V1.1 predictions saved successfully.
File: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v1_1_predictions.csv


In [55]:
from pathlib import Path

# Change this name only if your five-fold dataset variable
# has a different name.
v1_1_fold_datasets = v11_fold_datasets

# Features required by V1.1
columns_to_save = [
    "user_id",
    "order_id",
    "correct",
    "overall_smoothed_accuracy"
]

# Create the output directory
feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_1_overall_history"
    / "v1.1"
)

feature_dir.mkdir(parents=True, exist_ok=True)

# Save training and validation data for all five folds
for fold_number, (train_df, val_df) in enumerate(
    v1_1_fold_datasets, start=1
):
    train_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_train.csv",
        index=False
    )

    val_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_validation.csv",
        index=False
    )

    print(f"Fold {fold_number} saved.")

print("\nV1.1 feature files saved successfully!")
print("Location:", feature_dir)

KeyError: "['overall_smoothed_accuracy'] not in index"

In [ ]:
print("Available variables containing 'fold':")

for name, value in list(globals().items()):
    if "fold" in name.lower() and isinstance(value, list):
        print(f"\nVariable: {name}")

        if len(value) > 0 and isinstance(value[0], tuple):
            train_df, val_df = value[0]

            print("Training columns:", train_df.columns.tolist())
            print("Validation columns:", val_df.columns.tolist())

Available variables containing 'fold':


RuntimeError: dictionary changed size during iteration

In [ ]:
print("Available variables containing 'fold':")

for name, value in list(globals().items()):
    if "fold" in name.lower() and isinstance(value, list):
        print(f"\nVariable: {name}")

        if len(value) > 0 and isinstance(value[0], tuple):
            train_df, val_df = value[0]

            print("Training columns:", train_df.columns.tolist())
            print("Validation columns:", val_df.columns.tolist())

Available variables containing 'fold':

Variable: fold_datasets
Training columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count']
Validation columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count']

Variable: v11_fold_datasets
Training columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count', 'previous_interactions', 'previous_correct', 'global_historical_prior', 'overall_smoothed_accuracy', 'log_previous_interactions']
Validation columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count', 'previous_interactions', 'previous_correct', 'global_historical_prior', 'overall_smoothed_accuracy', 'log_previous_interactions']

Variable: v1_1_fold_datasets
Training columns: ['user_id', 'order_id', 'problem_id', 'correct', 'skill_id', 'skill_set', 'attempt_count']
Validation columns: ['user_id', 'order_id', 'problem_id', 'correct',

In [ ]:
feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_1_overall_history"
    / "v1.1"
)

feature_dir.mkdir(parents=True, exist_ok=True)

columns_to_save = [
    "user_id",
    "order_id",
    "correct",
    "overall_smoothed_accuracy"
]

for fold_number, (train_df, val_df) in enumerate(
    v11_fold_datasets, start=1
):
    train_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_train.csv",
        index=False
    )

    val_df[columns_to_save].to_csv(
        feature_dir / f"fold_{fold_number}_validation.csv",
        index=False
    )

    print(f"Fold {fold_number} saved.")

print("\nV1.1 features saved successfully!")
print("Location:", feature_dir)

Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.

V1.1 features saved successfully!
Location: c:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_1_overall_history\v1.1
